{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# 🤖 Comprehensive Guide to AI Agents\n",
    "This notebook covers Agent Fundamentals, Prompt Engineering, Tool Calling, Memory, Planning, Multi-Agent Systems, and MCP (Model Context Protocol).\n",
    "\n",
    "All code cells use standard Python and require no external dependencies. They simulate real-world agent architectures for educational clarity."
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 1. Agent Fundamentals\n",
    "\n",
    "### What is an AI Agent?\n",
    "An AI Agent is an autonomous system that perceives its environment, reasons about goals, executes actions via tools, and iteratively adapts based on feedback. Unlike static models, agents close the loop between perception and action.\n",
    "\n",
    "### Agent vs LLM\n",
    "- **LLM**: Passive text generator. Outputs are deterministic/stochastic based on prompt. No state, no tool use, no iteration.\n",
    "- **Agent**: LLM + Memory + Tools + Planning + Execution Loop. Maintains state, interacts with external systems, self-corrects, and pursues multi-step objectives.\n",
    "\n",
    "### Agent Lifecycle\n",
    "`Observation → Reasoning → Action → Feedback → (Loop)`\n",
    "1. **Observation**: Ingests user input, environment state, or tool outputs.\n",
    "2. **Reasoning**: Plans, decomposes tasks, selects tools, or generates hypotheses.\n",
    "3. **Action**: Executes tool calls, API requests, or code.\n",
    "4. **Feedback**: Evaluates results, updates memory, triggers self-correction or next step.\n",
    "\n",
    "### Key Paradigms\n",
    "- **ReAct (Reason + Act)**: Interleaves `Thought`, `Action`, `Observation` steps. Prevents hallucination by grounding reasoning in tool outputs.\n",
    "- **Reflection & Self-Correction**: Agent critiques its own output, identifies logical/tool-use errors, and retries with adjusted prompts.\n",
    "- **Multi-step Reasoning**: Chains intermediate conclusions to solve complex problems where a single prompt fails."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Simulating the Agent Lifecycle Loop\n",
    "class SimpleAgent:\n",
    "    def __init__(self, max_steps=5):\n",
    "        self.memory = []\n",
    "        self.max_steps = max_steps\n",
    "\n",
    "    def observe(self, input_data):\n",
    "        self.memory.append(f\"[Observation] {input_data}\")\n",
    "        return input_data\n",
    "\n",
    "    def reason(self, observation):\n",
    "        # Simulated LLM reasoning\n",
    "        return f\"[Thought] Based on '{observation}', I should calculate the result.\"\n",
    "\n",
    "    def act(self, thought):\n",
    "        return \"[Action] execute_tool(params)\"\n",
    "\n",
    "    def feedback(self, result):\n",
    "        self.memory.append(f\"[Feedback] Tool returned: {result}\")\n",
    "        return result == \"SUCCESS\"\n",
    "\n",
    "    def run(self, task):\n",
    "        obs = self.observe(task)\n",
    "        for i in range(self.max_steps):\n",
    "            thought = self.reason(obs)\n",
    "            action = self.act(thought)\n",
    "            obs = \"SUCCESS\" if i == 0 else \"RETRY\"\n",
    "            done = self.feedback(obs)\n",
    "            print(f\"Step {i+1}: {thought}\\n{action}\\nResult: {obs}\")\n",
    "            if done:\n",
    "                break\n",
    "\n",
    "agent = SimpleAgent()\n",
    "agent.run(\"Calculate 2024 revenue growth\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 2. Prompt Engineering for Agents\n",
    "\n",
    "### System Prompts & Tool Instructions\n",
    "- Define role, constraints, output format, and available tools.\n",
    "- Explicitly state when to call tools vs when to answer directly.\n",
    "\n",
    "### Few-Shot Prompting\n",
    "Provide 2-3 examples of correct tool usage, reasoning traces, and structured outputs to ground the LLM.\n",
    "\n",
    "### Chain of Thought (CoT) & ReAct Prompting\n",
    "- **CoT**: `Let's think step by step...`\n",
    "- **ReAct**: Enforces `Thought: ...\\nAction: ...\\nObservation: ...` format.\n",
    "\n",
    "### Planning Prompts\n",
    "Ask the agent to generate a step-by-step plan *before* execution. Reduces hallucination and improves tool sequencing.\n",
    "\n",
    "### Structured Outputs\n",
    "- **JSON/XML**: Enforce schemas for reliable parsing.\n",
    "- **Function Calling**: Native LLM feature that returns structured tool invocation objects instead of free text."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import json\n",
    "\n",
    "SYSTEM_PROMPT = \"\"\"You are a research agent. Always follow this format:\n",
    "1. Generate a step-by-step plan.\n",
    "2. Use tools when external data is needed.\n",
    "3. Output final results in JSON.\n",
    "\"\"\"\n",
    "\n",
    "FEW_SHOT_EXAMPLES = \"\"\"\n",
    "User: What's the weather in Tokyo?\n",
    "Thought: I need current weather data. I will call weather_api.\n",
    "Action: weather_api(location=\"Tokyo\")\n",
    "Observation: {\"temp\": 22, \"condition\": \"Clear\"}\n",
    "Answer: Tokyo is currently 22°C and clear.\n",
    "\"\"\"\n",
    "\n",
    "# JSON Schema Enforcement Example\n",
    "def validate_json_output(llm_response: str) -> dict:\n",
    "    try:\n",
    "        data = json.loads(llm_response)\n",
    "        required = [\"status\", \"steps\", \"result\"]\n",
    "        if all(k in data for k in required):\n",
    "            return data\n",
    "        raise ValueError(\"Missing required keys\")\n",
    "    except json.JSONDecodeError as e:\n",
    "        return {\"error\": \"Invalid JSON\", \"raw\": llm_response}\n",
    "\n",
    "mock_llm = '{\"status\": \"success\", \"steps\": [1, 2], \"result\": 42}'\n",
    "print(\"Parsed:\", validate_json_output(mock_llm))"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 3. Tool Calling\n",
    "\n",
    "### Function Calling\n",
    "LLMs output structured payloads matching predefined tool schemas. The runtime parses and executes them.\n",
    "\n",
    "### Tool Selection & Routing\n",
    "- **Rule-based**: Keyword/regex matching.\n",
    "- **Semantic**: Embedding similarity between user intent and tool descriptions.\n",
    "- **LLM-based**: Let the model choose from a registry.\n",
    "\n",
    "### Tool Execution & Error Handling\n",
    "Tools must return standardized responses (success, error, partial). Agents should retry with fallback parameters or escalate to human-in-the-loop."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "class ToolRegistry:\n",
    "    def __init__(self):\n",
    "        self.tools = {}\n",
    "\n",
    "    def register(self, name, description, func):\n",
    "        self.tools[name] = {\"description\": description, \"func\": func}\n",
    "\n",
    "    def route(self, intent: str) -> str:\n",
    "        # Simulated LLM routing decision\n",
    "        return \"search_web\" if \"search\" in intent.lower() else \"calculate\"\n",
    "\n",
    "    def execute(self, tool_name: str, **kwargs):\n",
    "        if tool_name not in self.tools:\n",
    "            return {\"error\": f\"Tool {tool_name} not found\"}\n",
    "        try:\n",
    "            return self.tools[tool_name][\"func\"](**kwargs)\n",
    "        except Exception as e:\n",
    "            return {\"error\": str(e)}\n",
    "\n",
    "registry = ToolRegistry()\n",
    "registry.register(\"search_web\", \"Search the internet\", lambda q: {\"results\": [\"Doc A\", \"Doc B\"]})\n",
    "registry.register(\"calculate\", \"Evaluate math\", lambda expr: {\"result\": eval(expr)})\n",
    "\n",
    "chosen = registry.route(\"Find info about AI safety\")\n",
    "print(\"Selected:\", chosen)\n",
    "print(\"Output:\", registry.execute(chosen, q=\"AI safety\"))"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 4. Agent Memory\n",
    "\n",
    "### Short-Term Memory\n",
    "- **Conversation History**: Rolling window of recent turns.\n",
    "- **Current Task**: Active goal, subtasks, state variables.\n",
    "- **Current Context**: Tool outputs, environment snapshots.\n",
    "\n",
    "### Long-Term Memory\n",
    "- **Preferences**: User tone, formatting choices, recurring constraints.\n",
    "- **User Facts**: Demographics, roles, known data.\n",
    "- **Previous Interactions**: Past tasks, solutions, failures.\n",
    "\n",
    "### Memory Concepts\n",
    "- **Episodic Memory**: Stores specific past events/sessions.\n",
    "- **Semantic Memory**: Stores generalized knowledge extracted from episodes.\n",
    "- **Retrieval**: Vector search, keyword filtering, or LLM reranking.\n",
    "- **Update**: Summarization, deduplication, conflict resolution, periodic consolidation."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "from datetime import datetime\n",
    "import uuid\n",
    "\n",
    "class AgentMemory:\n",
    "    def __init__(self):\n",
    "        self.short_term = []  # Recent turns\n",
    "        self.long_term = {\"facts\": {}, \"episodic\": [], \"semantic\": {}}\n",
    "\n",
    "    def add_short_term(self, role: str, content: str):\n",
    "        self.short_term.append({\"role\": role, \"content\": content, \"ts\": datetime.now().isoformat()})\n",
    "        if len(self.short_term) > 10:\n",
    "            self.short_term.pop(0)  # Sliding window\n",
    "\n",
    "    def store_fact(self, key: str, value: str):\n",
    "        self.long_term[\"facts\"][key] = value\n",
    "\n",
    "    def store_episode(self, session_id: str, summary: str):\n",
    "        self.long_term[\"episodic\"].append({\"id\": session_id, \"summary\": summary, \"ts\": datetime.now().isoformat()})\n",
    "\n",
    "    def retrieve_semantic(self, query: str) -> list:\n",
    "        # Simulated vector/keyword retrieval\n",
    "        return [v for k, v in self.long_term[\"semantic\"].items() if query.lower() in k.lower()]\n",
    "\n",
    "    def consolidate(self):\n",
    "        # Pseudo: Run LLM summarization on recent episodes, update semantic memory\n",
    "        for ep in self.long_term[\"episodic\"][-3:]:\n",
    "            self.long_term[\"semantic\"][f\"topic_{uuid.uuid4().hex[:6]}\"] = ep[\"summary\"]\n",
    "\n",
    "mem = AgentMemory()\n",
    "mem.add_short_term(\"user\", \"I prefer JSON responses\")\n",
    "mem.store_fact(\"user_name\", \"Alice\")\n",
    "mem.store_episode(\"sess_01\", \"Calculated Q3 metrics\")\n",
    "mem.consolidate()\n",
    "print(\"Long-term facts:\", mem.long_term[\"facts\"])\n",
    "print(\"Semantic index:\", list(mem.long_term[\"semantic\"].keys()))"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 5. Planning & Multi-Agent Systems\n",
    "\n",
    "### Task Decomposition\n",
    "Break complex goals into atomic, executable subtasks. Use dependency graphs to order execution.\n",
    "\n",
    "### Planning Algorithms\n",
    "- **ReAct**: Interleaved reasoning/action. Good for dynamic environments.\n",
    "- **Plan-and-Execute**: Generate full plan upfront → execute sequentially → adapt if failed.\n",
    "- **Tree of Thoughts (ToT)**: Explores multiple reasoning branches, scores them, backtracks.\n",
    "- **Graph of Thoughts (GoT)**: DAG-based reasoning. Allows merging, looping, and parallel branches.\n",
    "- **Reflexion**: Post-action self-evaluation. Stores verbal reinforcement signals to avoid repeating mistakes.\n",
    "\n",
    "### Multi-Agent Systems\n",
    "- **Communication**: Message buses, shared state, or direct function calls.\n",
    "- **Coordination**: Role assignment, consensus protocols, voting, or lock-step execution.\n",
    "- **Delegation**: Manager agent splits work, assigns to specialists.\n",
    "- **Orchestration**: Central controller routes, monitors, and aggregates results."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "class MultiAgentOrchestrator:\n",
    "    def __init__(self):\n",
    "        self.agents = {\"researcher\": [], \"coder\": [], \"reviewer\": []}\n",
    "        self.message_bus = []\n",
    "\n",
    "    def delegate(self, task: str, roles: list):\n",
    "        print(f\"[Orchestrator] Decomposing: {task}\")\n",
    "        for role in roles:\n",
    "            self.agents[role].append(task)\n",
    "            self.message_bus.append({\"from\": \"orchestrator\", \"to\": role, \"payload\": task})\n",
    "\n",
    "    def coordinate(self):\n",
    "        results = {}\n",
    "        for role, tasks in self.agents.items():\n",
    "            # Simulated parallel execution\n",
    "            results[role] = f\"{role} completed {len(tasks)} tasks\"\n",
    "        return results\n",
    "\n",
    "orch = MultiAgentOrchestrator()\n",
    "orch.delegate(\"Build a weather dashboard\", [\"researcher\", \"coder\", \"reviewer\"])\n",
    "print(\"Messages:\", orch.message_bus)\n",
    "print(\"Results:\", orch.coordinate())"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 6. MCP (Model Context Protocol)\n",
    "\n",
    "### Architecture\n",
    "MCP is an open standard that decouples LLMs from tools. It defines a universal JSON-RPC 2.0 interface for tool discovery, invocation, and context sharing.\n",
    "\n",
    "### MCP Client\n",
    "The agent runtime or LLM wrapper. Handles:\n",
    "- Connecting to MCP servers\n",
    - Sending tool calls\n",
    - Parsing structured responses\n",
    "\n",
    "### MCP Server\n",
    "Hosts tools/resources. Exposes:\n",
    "- `tools/list`: Returns tool schemas\n",
    "- `tools/call`: Executes tool with args\n",
    "- `resources/read`: Fetches context/data\n",
    "\n",
    "### Tool Discovery & Invocation\n",
    "Clients query servers at startup, cache schemas, and invoke tools via standardized RPC payloads. Eliminates vendor lock-in and simplifies multi-framework tool integration."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import json\n",
    "\n",
    "class MCPServer:\n",
    "    def __init__(self):\n",
    "        self.tools = [\n",
    "            {\"name\": \"get_stock_price\", \"description\": \"Fetch current stock price\", \"inputSchema\": {\"type\": \"object\", \"properties\": {\"symbol\": {\"type\": \"string\"}}}}\n",
    "        ]\n",
    "\n",
    "    def handle_request(self, rpc: dict) -> dict:\n",
    "        method = rpc.get(\"method\")\n",
    "        params = rpc.get(\"params\", {})\n",
    "        \n",
    "        if method == \"tools/list\":\n",
    "            return {\"jsonrpc\": \"2.0\", \"id\": rpc[\"id\"], \"result\": {\"tools\": self.tools}}\n",
    "        elif method == \"tools/call\":\n",
    "            name = params.get(\"name\")\n",
    "            args = params.get(\"arguments\", {})\n",
    "            if name == \"get_stock_price\":\n",
    "                return {\"jsonrpc\": \"2.0\", \"id\": rpc[\"id\"], \"result\": {\"content\": [{\"type\": \"text\", \"text\": f\"${args.get('symbol')}: $142.50\"}]}}\n",
    "        return {\"jsonrpc\": \"2.0\", \"id\": rpc[\"id\"], \"error\": {\"code\": -32601, \"message\": \"Method not found\"}}\n",
    "\n",
    "class MCPClient:\n",
    "    def __init__(self, server: MCPServer):\n",
    "        self.server = server\n",
    "        self.discovered_tools = []\n",
    "\n",
    "    def discover(self):\n",
    "        resp = self.server.handle_request({\"jsonrpc\": \"2.0\", \"id\": 1, \"method\": \"tools/list\", \"params\": {}})\n",
    "        self.discovered_tools = resp[\"result\"][\"tools\"]\n",
    "        return self.discovered_tools\n",
    "\n",
    "    def invoke(self, tool_name: str, args: dict):\n",
    "        resp = self.server.handle_request({\n",
    "            \"jsonrpc\": \"2.0\", \"id\": 2, \"method\": \"tools/call\",\n",
    "            \"params\": {\"name\": tool_name, \"arguments\": args}\n",
    "        })\n",
    "        return resp[\"result\"][\"content\"][0][\"text\"]\n",
    "\n",
    "server = MCPServer()\n",
    "client = MCPClient(server)\n",
    "print(\"Discovered:\", client.discover())\n",
    "print(\"Invocation:\", client.invoke(\"get_stock_price\", {\"symbol\": \"AAPL\"}))"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 📌 Summary & Next Steps\n",
    "- **Agents** close the perception-action loop using memory, tools, and planning.\n",
    "- **Prompt Engineering** structures reasoning, enforces formats, and guides tool use.\n",
    "- **Tool Calling** requires robust routing, execution, and error recovery.\n",
    "- **Memory** scales from sliding windows to vector-backed episodic/semantic stores.\n",
    "- **Planning** algorithms (ReAct, ToT, Reflexion) solve complex, multi-step tasks.\n",
    "- **MCP** standardizes tool integration across frameworks and models.\n",
    "\n",
    "To build production agents:\n",
    "1. Use framework-agnostic abstractions (LangGraph, LlamaIndex, CrewAI, or custom MCP clients).\n",
    "2. Implement structured output validation + retry loops.\n",
    "3. Add observability (tracing, latency, tool success rates).\n",
    "4. Evaluate with agent-specific benchmarks (AgentBench, SWE-bench, WebArena)."
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "name": "python",
   "version": "3.10.0"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 5
}